In [ ]:

! pip install unsloth datasets transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.8/79.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.4/30.4 MB 53.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 75.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 945.8/945.8 kB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 85.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 68.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 76.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.5/230.5 kB 24.2 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing ins

In [ ]:
import os
import pandas as pds
import unsloth
import torch
import transformers
import datasets
import gc
import json
import time
from pathlib import Path


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: xformers is installed but cannot load with torch 2.11.0+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from unsloth import FastLanguageModel, is_bf16_supported

print("done")

done


In [ ]:
from google.colab import drive
DATA_DIR = drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from datasets import load_dataset


trn_data = "/content/drive/MyDrive/FINE-TUNE/DATASET/dataset_train.jsonl"
tst_data = "/content/drive/MyDrive/FINE-TUNE/DATASET/dataset_test.jsonl"
vld_data = "/content/drive/MyDrive/FINE-TUNE/DATASET/dataset_validation.jsonl"

data_files = {
    "train": trn_data,
    "validation": vld_data,
    "test": tst_data,
}
datasets = load_dataset("json", data_files=data_files)

print(datasets)

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['messages', 'metadata'],
        num_rows: 4605
    })
    validation: Dataset({
        features: ['messages', 'metadata'],
        num_rows: 499
    })
    test: Dataset({
        features: ['messages', 'metadata'],
        num_rows: 499
    })
})


In [ ]:

MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"
MAX_SEQ_LENGTH= 2048


model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = None,
    load_in_4bit = True
)

==((====))==  Unsloth 2026.10.6: Fast Qwen3 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [ ]:
#  saving some examples from base model to compare it with fine-tuned one

from transformers import pipeline

# 1. Extract the 3 splits from your `datasets` DatasetDict
train_dataset = datasets["train"]
val_dataset   = datasets["validation"]
test_dataset  = datasets["test"]

# 2. Pick 5 unseen test rows for our Before vs. After comparison
eval_test_slice = test_dataset.select(range(5))

def get_answers(model, tokenizer):
    FastLanguageModel.for_inference(model)
    pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

    prompts = [r["messages"][:-1] for r in eval_test_slice]
    outputs = pipe(prompts, max_new_tokens=256, return_full_text=False, do_sample=True, temperature=0.7)

    del pipe
    gc.collect()
    torch.cuda.empty_cache()
    return [out[0]["generated_text"].strip() for out in outputs]

print("Generating Base Model baseline answers...")
base_model_answers = get_answers(model, tokenizer)
print(f"\nSample 1 Base Answer:\n{base_model_answers[0][:250]}...")

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'pad_token_id', 'temperature', 'cache_implementation', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Generating Base Model baseline answers...


Both `max_new_tokens` (=256) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
Both `max_new_tokens` (=256) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=262144) seem to have been set. `max_new_tokens


Sample 1 Base Answer:
A **500 Internal Server Error** due to an **"Unhandled SQLAlchemy lazy load outside session scope"** typically occurs when you're trying to access a relationship or attribute on a model object (e.g., `obj.user.name`) that was loaded outside of an act...


In [ ]:
#  Adding LoRA adapter
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha = 16,
    lora_dropout=0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
    use_rslora=False,
    loftq_config=None

)

Unsloth 2026.10.6 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


In [ ]:
# 1. Switch model back into training mode
FastLanguageModel.for_training(model)

# 2. Convert the 'messages' list of dicts into a flat Qwen3 ChatML string
def format_chat_prompts(batch):
    texts = [
        tokenizer.apply_chat_template(
            convo,
            tokenize=False,               # Keep as readable text; SFTTrainer tokenizes it later
            add_generation_prompt=False   # False because the assistant's answer is already inside convo!
        )
        for convo in batch["messages"]
    ]
    return {"text": texts}

train_dataset = train_dataset.map(format_chat_prompts, batched=True)
val_dataset   = val_dataset.map(format_chat_prompts, batched=True)

# 4. Check how many parameters we are actually training & preview Row 0
model.print_trainable_parameters()
print("\n--- FORMATTED TRAINING TEXT PREVIEW ---")
print(train_dataset[0]["text"][:350])

Map:   0%|          | 0/4605 [00:00<?, ? examples/s]

Map:   0%|          | 0/499 [00:00<?, ? examples/s]

trainable params: 33,030,144 || all params: 4,055,498,240 || trainable%: 0.8145

--- FORMATTED TRAINING TEXT PREVIEW ---
<|im_start|>user
Tune Kafka consumer lag in order pipeline during peak traffic. Give tuning guide.<|im_end|>
<|im_start|>assistant
<think>

</think>

Measure lag per partition, processing time, fetch rate, GC pauses. Scale partitions and consumer instances matching key ordering constraints. Increase fetch size, reduce poll interval overhead, batch 


In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers.trainer_utils import get_last_checkpoint

CHECKPOINT_DIR = "/content/drive/MyDrive/FINE-TUNE/qwen3_4b_checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

training_args = SFTConfig(
    output_dir = CHECKPOINT_DIR,
    dataset_text_field = "text",
    max_seq_length = MAX_SEQ_LENGTH,      # 2048

    # 1. Your Tuned T4 Batching (Effective Batch Size = 6 * 2 = 12)
    packing = True,
    per_device_train_batch_size = 6,      # Safe ~11GB VRAM footprint on T4
    gradient_accumulation_steps = 2,      # 6 * 2 = 12 Effective Batch Size
    dataloader_num_workers = 2,           # Matches Colab's 2 vCPU threads cleanly
    dataloader_pin_memory = True,

    # 2. Learning Schedule & 8-Bit Optimizer
    num_train_epochs = 3,
    learning_rate = 2e-4,
    warmup_ratio = 0.05,
    lr_scheduler_type = "cosine",
    optim = "adamw_8bit",

    # 3. Precision (fp16 on Tesla T4)
    fp16 = not is_bf16_supported(),
    bf16 = is_bf16_supported(),

    # 4. Logging, Validation & Google Drive Checkpoints
    logging_steps = 5,
    eval_strategy = "steps",
    eval_steps = 50,
    save_strategy = "steps",
    save_steps = 50,
    save_total_limit = 2,
    seed = 42,
    report_to = "none",
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = val_dataset,
    args = training_args,
)

# Auto-resume from Google Drive if Colab ever disconnects
last_checkpoint = get_last_checkpoint(CHECKPOINT_DIR)
if last_checkpoint is not None:
    print(f"🔄 Resuming from checkpoint: {last_checkpoint}")
    trainer_stats = trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    print("🚀 Starting fresh packed training run...")
    trainer_stats = trainer.train()

Unsloth: transformers renamed `warmup_ratio` to `warmup_steps`. Forwarding your value to `warmup_steps` - update your code when convenient. If you also passed `warmup_steps` as 0.1, that is its default here and cannot be distinguished from leaving it unset, so `warmup_ratio` was used; drop `warmup_ratio` to keep it.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/4605 [00:00<?, ? examples/s]

Unsloth: Packing train dataset (num_proc=2):   0%|          | 0/4605 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/499 [00:00<?, ? examples/s]

Unsloth: Packing eval dataset (num_proc=2):   0%|          | 0/499 [00:00<?, ? examples/s]

🦥 Unsloth: Packing enabled - training is >2x faster and uses less VRAM!
🚀 Starting fresh packed training run...


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 651 | Num Epochs = 3 | Total steps = 165
O^O/ \_/ \    Batch size per device = 6 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (6 x 2 x 1) = 12
 "-____-"     Trainable parameters = 33,030,144 of 4,055,498,240 (0.81% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
50,1.406933,2.471723
100,1.447318,2.217850
150,1.478399,2.156769
165,1.252927,2.155693


Unsloth: Not an error, but `num_items_in_batch` could not be counted for Qwen3ForCausalLM's labels, so each micro-batch loss is a mean.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/FINE-TUNE/qwen3_4b_checkpoints/checkpoint-50/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/FINE-TUNE/qwen3_4b_checkpoints/checkpoint-100/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/FINE-TUNE/qwen3_4b_checkpoints/checkpoint-150/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/FINE-TUNE/qwen3_4b_checkpoints/checkpoint-165/tokenizer_config.json.


In [ ]:
from IPython.display import display

# 1. Generate answers using the newly trained model (automatically switches to inference mode!)
print("Generating Fine-Tuned Model answers on the same 5 unseen test questions...")
fine_tuned_answers = get_answers(model, tokenizer)

# 2. Extract the User Question and Target Assistant Answer from our 5 test rows
questions = [r["messages"][0]["content"] for r in eval_test_slice]
target_ground_truth = [r["messages"][-1]["content"] for r in eval_test_slice]

# 3. Build a clean side-by-side comparison table (using `pds` from your imports!)
pds.set_option("display.max_colwidth", None)  # Prevents Pandas from cutting off long text with "..."

comparison_df = pds.DataFrame({
    "Question": questions,
    "Target Answer (Dataset)": target_ground_truth,
    "Base Model (Before)": base_model_answers,
    "Fine-Tuned (After)": fine_tuned_answers,
})

print("\n--- QUALITATIVE COMPARISON: BEFORE VS. AFTER ---")
display(comparison_df)

Both `max_new_tokens` (=256) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generating Fine-Tuned Model answers on the same 5 unseen test questions...


Both `max_new_tokens` (=256) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_


--- QUALITATIVE COMPARISON: BEFORE VS. AFTER ---


,Question,Target Answer (Dataset),Base Model (Before),Fine-Tuned (After)
0,Debug backend 500 error from unhandled SQLAlchemy lazy load outside session scope.,"DetachedInstanceError means accessing lazy relationship after session closed. Fix by eager loading with selectinload joinedload in query, or mapping to DTO inside session before return. Avoid lazy in async where greenlet issues arise. Add repository method returning fully loaded DTO, close session via dependency yield finally. Log query path, enable SQL echo in staging. Test accessing relationships after return expecting no IO, cover nested relations, empty collections. Profile query count with assert_num_queries, avoid N plus one. Document loading strategy per endpoint, when to split endpoints versus expand param, and clean architecture rule keeping ORM models in adapter only with domain entities detached and serializable.","A **500 Internal Server Error** due to an **""Unhandled SQLAlchemy lazy load outside session scope""** typically occurs when you're trying to access a relationship or attribute on a model object (e.g., `obj.user.name`) that was loaded outside of an active SQLAlchemy session — especially when using **lazy loading** (like `lazy='dynamic'`, `lazy='select'`, or `lazy='joined'`).\n\nThis is a common issue in Flask-SQLAlchemy or FastAPI apps when you access related objects after the session has been closed or when you're accessing data in a view or template without proper session context.\n\n---\n\n### 🔍 Root Cause\n\nWhen you define a relationship with `lazy='select'` or `lazy='dynamic'`, SQLAlchemy will **defer** the query until the attribute is accessed. If you try to access that attribute **outside of the session scope** (e.g., in a route, a template, or a background task), SQLAlchemy will attempt to execute the query — but fails because there's no active session.\n\n> Example:\n> ```python\n> user = User.query.get(1)\n> print(user.orders) # This triggers lazy loading\n> ```\n> If `user` was fetched from a different session or","</tool_call>\n\n</tool_call>\n\nLazy loading with association backref without explicit session flush or outer join raises SessionInvalidException when ORM tries to resolve foreign keys. Ensure all association queries use explicit join or selectinload with proper session context. Example:\nfrom sqlalchemy.orm import selectinload\nwith Session() as session:\n users = session.execute(\n select(User).options(selectinload(User.orders))\n ).scalars().all()\n# Never do this:\n# user = session.get(User, 1)\n# orders = user.orders # lazy load fails\nAdd session context manager and use selectinload for eager loading. Enable ORM logging to inspect query plans. Add test asserting no SessionInvalidException. Validate foreign key constraints in migrations. Monitor 500 rate for lazy load failures. Document eager loading patterns and when to use lazy. This prevents session leakage and improves query efficiency in web APIs."
1,Prevent prompt injection via third party webhook content in support bot.,"Treat webhook payload as untrusted data, never system instructions. Isolate with delimiters and instruct model to ignore embedded directives. Sanitize HTML and strip control tokens before insertion. Enforce allowlisted tool actions with human approval for sensitive ops. Log raw payloads for audit and run injection detectors. Test with known jailbreak strings and indirect instruction smuggling. Apply least privilege API scopes for bot actions. Add output validation blocking exfiltration patterns. For backends, version prompts and review changes. This defense in depth reduces risk of data theft or unauthorized actions. It acknowledges models cannot perfectly distinguish data from instructions, so architectural guardrails must constrain impact even when injections bypass filters.","To **prevent prompt injection via third-party webhook content in a support bot**, you must implement a robust, multi-layered defense strategy that focuses on **input valida

In [ ]:
! pip install evaluate
import evaluate

print("\n--- CALCULATING QUANTITATIVE ROUGE SCORES ---")
rouge = evaluate.load("rouge")

# 1. Compute ROUGE for Base Model vs Target Dataset
base_rouge = rouge.compute(
    predictions=base_model_answers,
    references=target_ground_truth
)

# 2. Compute ROUGE for Fine-Tuned Model vs Target Dataset
ft_rouge = rouge.compute(
    predictions=fine_tuned_answers,
    references=target_ground_truth
)

# 3. Build a clean score comparison table using Pandas
score_df = pds.DataFrame({
    "Metric": ["ROUGE-1 (Keywords)", "ROUGE-2 (Phrases)", "ROUGE-L (Structure)"],
    "Base Model": [
        f"{base_rouge['rouge1'] * 100:.2f}%",
        f"{base_rouge['rouge2'] * 100:.2f}%",
        f"{base_rouge['rougeL'] * 100:.2f}%"
    ],
    "Fine-Tuned": [
        f"{ft_rouge['rouge1'] * 100:.2f}%",
        f"{ft_rouge['rouge2'] * 100:.2f}%",
        f"{ft_rouge['rougeL'] * 100:.2f}%"
    ],
    "Improvement": [
        f"{(ft_rouge['rouge1'] - base_rouge['rouge1']) * 100:+.2f}%",
        f"{(ft_rouge['rouge2'] - base_rouge['rouge2']) * 100:+.2f}%",
        f"{(ft_rouge['rougeL'] - base_rouge['rougeL']) * 100:+.2f}%",
    ]
})

display(score_df)

# saving adapters to the google drive so that we can use the adapters later
ADAPTER_SAVE_PATH = "/content/drive/MyDrive/FINE-TUNE/qwen3_4b_lora_adapter"

print(f"\nSaving final LoRA adapter weights to: {ADAPTER_SAVE_PATH}")
model.save_pretrained(ADAPTER_SAVE_PATH)
tokenizer.save_pretrained(ADAPTER_SAVE_PATH)
print("✅ Training pipeline complete! LoRA adapters saved securely to Google Drive.")


--- CALCULATING QUANTITATIVE ROUGE SCORES ---


,Metric,Base Model,Fine-Tuned,Improvement
0,ROUGE-1 (Keywords),21.37%,30.64%,+9.26%
1,ROUGE-2 (Phrases),2.85%,6.29%,+3.43%
2,ROUGE-L (Structure),9.92%,18.43%,+8.51%



Saving final LoRA adapter weights to: /content/drive/MyDrive/FINE-TUNE/qwen3_4b_lora_adapter


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/FINE-TUNE/qwen3_4b_lora_adapter/tokenizer_config.json.


✅ Training pipeline complete! LoRA adapters saved securely to Google Drive.


In [ ]:
import shutil

# 1. Export to Colab's fast local SSD instead of Google Drive
LOCAL_GGUF_PATH = "/content/Qwen_PRIME4B_AI_gguf"

model.save_pretrained_gguf(
    LOCAL_GGUF_PATH,
    tokenizer,
    quantization_method = "q4_k_m"
)

# 2. Once finished, copy the finished GGUF folder safely to Google Drive
DRIVE_DEST = "/content/drive/MyDrive/FINE-TUNE/Qwen_PRIME4B_AI_gguf"
shutil.copytree(LOCAL_GGUF_PATH, DRIVE_DEST, dirs_exist_ok=True)
print("✅ GGUF successfully built locally and copied to Google Drive!")

Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/1.57k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/32.9k [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in /content/Qwen_PRIME4B_AI_gguf/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


...

model-00002-of-00002.safetensors: downloading bytes:           |  0.00B            



Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [04:55<00:00, 147.84s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)




Unsloth: Merging weights into 16bit:   0%|          | 0/2 [00:00<?, ?it/s]

Unsloth: Merging weights into 16bit:  50%|█████     | 1/2 [02:30<02:30, 150.05s/it]

Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [04:06<00:00, 123.17s/it]


Unsloth: Merge process complete. Saved to `/content/Qwen_PRIME4B_AI_gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b11696-mix-7baeb9a (app-b11696-mix-7baeb9a-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['/content/Qwen_PRIME4B_AI_gguf_gguf/Qwen3-4B-Instruct-2507.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF 

In [ ]:
import shutil

# Copy from the correct local folder where Unsloth saved the .gguf file
shutil.copytree("/content/Qwen_PRIME4B_AI_gguf_gguf", "/content/drive/MyDrive/FINE-TUNE/Qwen_PRIME4B_AI_gguf", dirs_exist_ok=True)
print("✅ GGUF model successfully copied to Google Drive!")

✅ GGUF model successfully copied to Google Drive!
